# Homothetic Tube MPC — Interactive Lane-Change Playground

Companion playground for **Homothetic Tube MPC at the Computational Cost of Rigid Tube MPC**
(M. Banitalebi Dehkordi, M. Mejari, D. Piga).

**Run all cells**, then use the controls that appear just below: choose a manoeuvre, the
speed, and the two uncertainty bounds, and press **Run**. The paper's figure is then drawn
progressively for both schemes — HT-MPC (proposed, scales its tube by
$\alpha_k = \|e_k\|_{P^{\star-1}}$) and RT-MPC (rigid, always uses the full $\Omega$): the lateral
position $e_y$, the nominal steering $v_{0|k}$ together with the set each scheme admits for it at
the current step, and the scale $\alpha_k$.

*The cell below holds the whole method; it is collapsed because you do not need to read it to
use the playground. The sections underneath explain the controls and the code.*

In [1]:
# ── Setup: install the few packages that may be missing ─────────────────────
import importlib.util, subprocess, sys

_missing = [p for p in ("cvxpy", "clarabel", "osqp", "scs", "plotly", "ipywidgets", "nbformat")
            if importlib.util.find_spec(p) is None]
if _missing:
    print("Installing:", " ".join(_missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

# ============================================================================
#  METHOD - verbatim copy of LatVeh_Benchmark.ipynb
# ============================================================================

import numpy as np
import scipy.linalg   as la
import scipy.signal   as sig
import scipy.stats    as stats
import scipy.optimize as opt
import cvxpy          as cp
import matplotlib.pyplot as plt
from matplotlib.lines   import Line2D
from matplotlib.patches import Patch
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
from dataclasses import dataclass
from itertools   import product as iprod

# ── matplotlib style (IEEE single-column) ─────────────────────────────────────
FP      = 8
ONE_COL = 3.5

plt.rcParams.update({
    "mathtext.fontset": "cm",
    "font.family":      "serif",
    "font.serif":       ["Times New Roman", "DejaVu Serif"],
    "font.size":         FP,    "axes.labelsize":   FP,
    "axes.titlesize":    FP,    "xtick.labelsize":  FP - 1,
    "ytick.labelsize":   FP - 1,"legend.fontsize":  FP - 1,
    "legend.framealpha": 0.90,  "legend.frameon":   True,
    "legend.edgecolor":  "0.80","lines.linewidth":  1.0,
    "axes.linewidth":    0.6,   "grid.linewidth":   0.4,
    "grid.alpha":        0.40,  "savefig.dpi":      300,
    "savefig.bbox":      "tight","pdf.fonttype":    42,
    "ps.fonttype":       42,
})

# Same fonts as the paper (IEEEtran: Times text, Computer Modern math).  With a
# LaTeX installation the labels are typeset by LaTeX with exactly those fonts;
# otherwise a Times clone is used for text and mathtext CM for math.
import io, shutil
_TEX_RC = {
    "text.usetex":         True,
    "font.serif":          ["Computer Modern Roman"],
    "text.latex.preamble": r"\renewcommand{\rmdefault}{ptm}",
}


def _latex_works():
    """True if matplotlib can typeset with LaTeX here (needs latex + cm-super)."""
    if shutil.which("latex") is None:
        return False
    try:
        with plt.rc_context(_TEX_RC):
            f = plt.figure();  f.text(.5, .5, r"test $v_{0|k}$")
            f.savefig(io.BytesIO(), format="pdf");  plt.close(f)
        return True
    except Exception:
        plt.close("all");  return False


USE_TEX = _latex_works()
if USE_TEX:
    plt.rcParams.update(_TEX_RC)
else:
    plt.rcParams.update({
        "font.serif": ["TeX Gyre Termes", "Nimbus Roman", "Times New Roman",
                       "Times", "DejaVu Serif"],
    })
print(f"  figure fonts: {'LaTeX (Times + CM math)' if USE_TEX else 'Times clone + CM mathtext'}")

C = {
    "p":      "#007E5C",     # HT-MPC  — green
    "r":      "#FF4500",     # RT-MPC  — orange-red
    "ref":    "#1D1D1DCA",   # reference — near-black
    "constr": "#A80000",     # constraints / road edges — dark red
}
LABELS = {"p": "HT-MPC", "r": "RT-MPC"}


# ══════════════════════════════════════════════════════════════════════════════
#  VEHICLE MODEL
# ══════════════════════════════════════════════════════════════════════════════

_m  = 1500.0;  _Iz = 2500.0
_lf = 1.2;     _lr = 1.6
_Cf = 60000.0; _Cr = 70000.0


def _Ac(vx):
    m, Iz, lf, lr, Cf, Cr = _m, _Iz, _lf, _lr, _Cf, _Cr
    return np.array([
        [0.,  vx,  1.,  0.],
        [0.,  0.,  0.,  1.],
        [0.,  0.,  -2.*(Cf+Cr)/(m*vx),
                   (-2.*(Cf*lf-Cr*lr)/(m*vx) - vx)],
        [0.,  0.,  -2.*(Cf*lf-Cr*lr)/(Iz*vx),
                   -2.*(Cf*lf**2+Cr*lr**2)/(Iz*vx)],
    ])


def _Bc():
    m, Iz, lf, Cf = _m, _Iz, _lf, _Cf
    return np.array([[0.], [0.], [2.*Cf/m], [2.*Cf*lf/Iz]])


def discretize(vx, Ts):
    sd = sig.StateSpace(_Ac(vx), _Bc(), np.eye(4),
                        np.zeros((4, 1))).to_discrete(Ts)   # ZOH
    return np.array(sd.A), np.array(sd.B)


def d_dvx(vx0, Ts, h=1e-4):
    """First-order sensitivities D_A = dA_d/dv_x and D_B = dB_d/dv_x."""
    Ap, Bp = discretize(vx0 + h, Ts)
    Am, Bm = discretize(vx0 - h, Ts)
    return (Ap - Am)/(2.*h), (Bp - Bm)/(2.*h)


# ══════════════════════════════════════════════════════════════════════════════
#  UNCERTAINTY MODEL  (Assumption 1)
# ══════════════════════════════════════════════════════════════════════════════

class UncertaintyModel:
    """
    Shapes Gamma_A = chi2_DA*Sigma_DA, Gamma_B = chi2_DB*Sigma_DB,
    Gamma_w = chi2_w*Sigma_w.  vec(.) is row-major here, consistent with
    Delta_A x = kron(I_n, x') vec(Delta_A).
    """
    def __init__(self, Sigma_DA, Sigma_DB, Sigma_w,
                 p_A, p_B, p_W, n, m, df_DA=1, df_DB=1):
        self.Sigma_DA = np.atleast_2d(Sigma_DA)
        self.Sigma_DB = np.atleast_2d(Sigma_DB)
        self.Sigma_w  = np.atleast_2d(Sigma_w)
        self.n = n;  self.m = m
        self.chi2_DA = float(stats.chi2.ppf(p_A, df=df_DA))
        self.chi2_DB = float(stats.chi2.ppf(p_B, df=df_DB))
        self.chi2_w  = float(stats.chi2.ppf(p_W, df=n))
        self.Gamma_w = self.chi2_w*self.Sigma_w
        print(f"  OK  chi2_DA={self.chi2_DA:.3f}  chi2_DB={self.chi2_DB:.3f}"
              f"  chi2_w={self.chi2_w:.3f}")

    def QA_at_x(self, x):
        L = np.kron(np.eye(self.n), x.reshape(1, -1))
        Q = L @ self.Sigma_DA @ L.T
        return 0.5*(Q + Q.T)

    def QB_at_u(self, u):
        L = np.kron(np.eye(self.n), u.reshape(1, -1))
        Q = L @ self.Sigma_DB @ L.T
        return 0.5*(Q + Q.T)


def cover_factors(vx0, Ts, A, B, Gamma_A, Gamma_B, dv_max,
                  n_grid=2001, margin=1.01):
    """
    Smallest factors kappa_A, kappa_B (times `margin`, never below 1) such that
    vec(A(vx0+dv) - A) in Ell(kappa_A Gamma_A) and vec(B(vx0+dv) - B) in
    Ell(kappa_B Gamma_B) for all dv on a fine grid of [-dv_max, dv_max], with
    the EXACT ZOH matrices (the shapes come from a first-order expansion).
    """
    GAi, GBi = la.inv(Gamma_A), la.inv(Gamma_B)
    qa = qb = 0.
    for dv in np.linspace(-dv_max, dv_max, n_grid):
        Ah, Bh = discretize(vx0 + dv, Ts)
        a = (Ah - A).flatten();  b = (Bh - B).flatten()   # row-major vec
        qa = max(qa, float(a @ GAi @ a));  qb = max(qb, float(b @ GBi @ b))
    kA, kB = max(1., margin*qa), max(1., margin*qb)
    print(f"  OK  exact-model check: max quad form A={qa:.4f}, B={qb:.4f} "
          f"-> inflate Gamma_A x{kA:.4f}, Gamma_B x{kB:.4f}")
    return kA, kB, qa, qb


# ══════════════════════════════════════════════════════════════════════════════
#  VERTEX BOUND (Lemma 2) + SDP (eq. Qmax)
# ══════════════════════════════════════════════════════════════════════════════

def _box_vertices(H, h):
    d = H.shape[1]
    assert H.shape[0] == 2*d
    assert np.allclose(H, np.vstack([np.eye(d), -np.eye(d)]))
    return np.array(list(iprod(*zip(-h[d:], h[:d]))))


def compute_Qmax(unc, Hx, h_x, Hu, h_u):
    n = unc.n
    def _sdp(mats):
        Qv = cp.Variable((n, n), symmetric=True)
        cp.Problem(cp.Minimize(cp.trace(Qv)),
                   [Qv >> M + 1e-9*np.eye(n) for M in mats]
                   ).solve(solver="SCS", verbose=False, eps=1e-8,
                           max_iters=15_000)
        if Qv.value is None:
            return mats[0] + 1e-6*np.eye(n)
        Q = 0.5*(Qv.value + Qv.value.T) + 1e-9*np.eye(n)
        # SCS solves the SDP only to a tolerance: lift Q so that Q >= M holds
        # exactly for every vertex (hypothesis of Lemma 2)
        gap = min(float(np.min(la.eigvalsh(Q - M))) for M in mats)
        if gap < 1e-9:
            Q = Q + (1e-9 - gap)*np.eye(n)
        return Q
    QA_max = _sdp([unc.QA_at_x(v) for v in _box_vertices(Hx, h_x)])
    QB_max = _sdp([unc.QB_at_u(v) for v in _box_vertices(Hu, h_u)])
    print(f"  OK  tr(Q^A_max)={np.trace(QA_max):.3e}  "
          f"tr(Q^B_max)={np.trace(QB_max):.3e}")
    return QA_max, QB_max


# ══════════════════════════════════════════════════════════════════════════════
#  RPI ELLIPSOID (Prop. 2) + CONTRACTION COEFFICIENT (Def. 1)
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class RPIData:
    P_star:     np.ndarray
    P_star_inv: np.ndarray
    c_S:        float
    tau_star:   float
    S_bar:      np.ndarray


def compute_rpi(A_K, unc, QA_max, QB_max):
    # optimal weights (Prop. 1) and S_bar (eq. Smax)
    G = [unc.chi2_DA*QA_max, unc.chi2_DB*QB_max, unc.Gamma_w]
    a = [float(np.trace(Gi)) for Gi in G]
    s = sum(np.sqrt(ai) for ai in a)
    mu = [np.sqrt(ai)/s for ai in a]
    S_bar = sum(Gi/mi for Gi, mi in zip(G, mu))
    S_bar = 0.5*(S_bar + S_bar.T)

    def _stein(tau):
        P = la.solve_discrete_lyapunov(A_K/np.sqrt(tau), S_bar/(1.-tau))
        return 0.5*(P + P.T)

    # golden-section search for tau* (tr(P_tau) is strictly convex)
    rho2 = float(np.max(np.abs(la.eigvals(A_K))))**2
    lo, hi = rho2 + 1e-8, 1. - 1e-8
    gr = (np.sqrt(5.) - 1.)/2.
    x1 = hi - gr*(hi - lo);  f1 = np.trace(_stein(x1))
    x2 = lo + gr*(hi - lo);  f2 = np.trace(_stein(x2))
    for _ in range(400):
        if hi - lo < 1e-12: break
        if f1 < f2:
            hi, x2, f2 = x2, x1, f1; x1 = hi - gr*(hi - lo); f1 = np.trace(_stein(x1))
        else:
            lo, x1, f1 = x1, x2, f2; x2 = lo + gr*(hi - lo); f2 = np.trace(_stein(x2))
    tau_star = 0.5*(lo + hi)
    P_star = _stein(tau_star)
    Pnh = np.real(la.inv(la.sqrtm(P_star)))
    c_S = float(np.max(np.real(la.eigvals(Pnh @ S_bar @ Pnh))))/(1. - tau_star)
    assert 0. < c_S < 1., f"c_S={c_S} outside (0,1) — check Assumption 2."
    print(f"  OK  tau*={tau_star:.4f}  c_S={c_S:.6f}  "
          f"sqrt(P*[0,0])={np.sqrt(P_star[0, 0]):.4f} m")
    return RPIData(P_star=P_star, P_star_inv=la.inv(P_star),
                   c_S=c_S, tau_star=tau_star, S_bar=S_bar)


def alpha_seq(alpha_k, c_S, N):
    """Closed-form scale sequence (Prop. 3)."""
    j = np.arange(N + 1, dtype=float)
    return np.sqrt(np.maximum(1. - (1. - c_S)**j*(1. - alpha_k**2), 0.))


# ══════════════════════════════════════════════════════════════════════════════
#  TIGHTENING (eq. support) + POLYTOPIC TERMINAL SET
# ══════════════════════════════════════════════════════════════════════════════

def support_values(K, P_star, Hx, Hu):
    s_x = np.array([np.sqrt(float(h @ P_star @ h)) for h in Hx])
    KPK = K @ P_star @ K.T
    s_u = np.array([np.sqrt(float(h @ KPK @ h)) for h in Hu])
    return s_x, s_u


def steady_state(A, B, C_out, r_ref):
    n_, m_ = A.shape[0], B.shape[1];  p_ = C_out.shape[0]
    r = np.atleast_1d(float(r_ref))
    M   = np.block([[A - np.eye(n_), B], [C_out, np.zeros((p_, m_))]])
    rhs = np.concatenate([np.zeros(n_), r])
    sol, *_ = np.linalg.lstsq(M, rhs, rcond=None)
    return sol[:n_], sol[n_:]


class TerminalSet:
    """
    Maximal positively invariant polytope of zeta+ = A_K zeta, with
    zeta = z - z_ref, under the rigid tightening (alpha = 1):
        Hx (z_ref + zeta)    <= h_x - s_x
        Hu (v_ref + K zeta)  <= h_u - s_u
    Rows G A_K^t, t = 0..t*, are fixed; the right-hand side depends on the
    setpoint.  t* is chosen so that the set is finitely determined for every
    setpoint passed in `setpoints` (Gilbert & Tan, 1991).
    """
    def __init__(self, A_K, K, Hx, h_x, Hu, h_u, s_x, s_u, setpoints,
                 t_max=200):
        self.G0 = np.vstack([Hx, Hu @ K])
        self.Hx, self.Hu = Hx, Hu
        self.bx = h_x - s_x;  self.bu = h_u - s_u
        t_star = 0
        for (zr, vr) in setpoints:
            g0 = self.rhs0(zr, vr)
            assert np.all(g0 > 0), "setpoint not in the interior of Z(1)"
            t_star = max(t_star, self._determinedness(A_K, g0, t_max))
        self.t_star = t_star
        self.G = np.vstack([self.G0 @ np.linalg.matrix_power(A_K, t)
                            for t in range(t_star + 1)])
        print(f"  OK  terminal set: t*={t_star}, {self.G.shape[0]} rows")

    def rhs0(self, z_ref, v_ref):
        return np.concatenate([self.bx - self.Hx @ z_ref,
                               self.bu - self.Hu @ v_ref])

    def rhs(self, z_ref, v_ref):
        return np.tile(self.rhs0(z_ref, v_ref), self.t_star + 1)

    def _determinedness(self, A_K, g0, t_max):
        Gs, gs = [self.G0], [g0]
        for t in range(t_max):
            Gt = np.vstack(Gs);  gt = np.concatenate(gs)
            Gn = self.G0 @ np.linalg.matrix_power(A_K, t + 1)
            redundant = True
            for i, row in enumerate(Gn):
                res = opt.linprog(-row, A_ub=Gt, b_ub=gt,
                                  bounds=[(None, None)]*A_K.shape[0],
                                  method="highs")
                if (not res.success) or (-res.fun > g0[i] + 1e-9):
                    redundant = False; break
            if redundant:
                return t
            Gs.append(Gn);  gs.append(g0)
        raise RuntimeError("terminal set not finitely determined")


# ══════════════════════════════════════════════════════════════════════════════
#  OCP  (Definition 2) — same QP for both schemes, different tightening
# ══════════════════════════════════════════════════════════════════════════════

class TubeOCP:
    def __init__(self, A, B, Hx, h_x, Hu, h_u, s_x, s_u, term, N,
                 Q_c, R_c, P_f, name):
        n = A.shape[0];  m = B.shape[1]
        self.N, self.h_x, self.h_u, self.s_x, self.s_u = N, h_x, h_u, s_x, s_u
        self.term = term
        self.z0_p = cp.Parameter(n);  self.zr_p = cp.Parameter(n)
        self.vr_p = cp.Parameter(m)
        self.hz_p = [cp.Parameter(Hx.shape[0]) for _ in range(N)]
        self.hv_p = [cp.Parameter(Hu.shape[0]) for _ in range(N)]
        self.gf_p = cp.Parameter(term.G.shape[0])
        z = cp.Variable((N + 1, n));  v = cp.Variable((N, m))
        self._z, self._v = z, v
        cost = 0.;  cons = [z[0] == self.z0_p]
        for j in range(N):
            cost += (cp.quad_form(z[j] - self.zr_p, Q_c)
                     + cp.quad_form(v[j] - self.vr_p, R_c))
            cons += [z[j+1] == A @ z[j] + B @ v[j],
                     Hx @ z[j] <= self.hz_p[j],
                     Hu @ v[j] <= self.hv_p[j]]
        cost += cp.quad_form(z[N] - self.zr_p, P_f)
        cons.append(term.G @ (z[N] - self.zr_p) <= self.gf_p)
        self.prob = cp.Problem(cp.Minimize(cost), cons)
        print(f"  OK  {name} OCP: N={N}, QP")

    def solve(self, z_k, z_ref, v_ref, aseq):
        self.z0_p.value = z_k;  self.zr_p.value = z_ref;  self.vr_p.value = v_ref
        for j in range(self.N):
            self.hz_p[j].value = self.h_x - aseq[j]*self.s_x
            self.hv_p[j].value = self.h_u - aseq[j]*self.s_u
        self.gf_p.value = self.term.rhs(z_ref, v_ref)
        # The OCP is a convex QP.  Clarabel (interior point) is used for
        # accuracy; OSQP is a fallback.
        try:
            self.prob.solve(solver=cp.CLARABEL, verbose=False)
        except cp.error.SolverError:
            pass
        if self.prob.status not in ("optimal", "optimal_inaccurate"):
            self.prob.solve(solver=cp.OSQP, eps_abs=1e-8, eps_rel=1e-8,
                            max_iter=200_000, polish=True, verbose=False)
        ok = self.prob.status in ("optimal", "optimal_inaccurate")
        return (self._z.value.copy(), self._v.value.copy(), True) if ok \
               else (None, None, False)


# ══════════════════════════════════════════════════════════════════════════════
#  SIMULATION (Algorithm 1)
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class SimResult:
    x:       np.ndarray
    z:       np.ndarray
    u:       np.ndarray
    v:       np.ndarray     # nominal input v_{0|k}
    alpha:   np.ndarray
    y_ref:   np.ndarray
    viol_x:  np.ndarray
    viol_u:  np.ndarray
    n_assum: int        # steps with alpha_k > 1 (assumption violation)
    cost:    float
    ok:      bool


def sample_w(rng, Sigma_w, Gamma_w_inv):
    """Gaussian sample conditioned on w in Ell(Gamma_w) (Assumption 1)."""
    while True:
        w = rng.multivariate_normal(np.zeros(Sigma_w.shape[0]), Sigma_w)
        if w @ Gamma_w_inv @ w <= 1.:
            return w


def run_mpc(A, B, A_hat, B_hat, K, C_out, Hx, h_x, Hu, h_u,
            unc, rpi, ocp, y_ref_sig, x0, N, Q_c, R_c,
            homothetic, rng_seed=None):
    rng = np.random.RandomState(rng_seed)
    Gw_inv = la.inv(unc.Gamma_w)
    T = len(y_ref_sig);  n = A.shape[0];  m = B.shape[1]
    Xt = np.zeros((T, n)); Zt = np.zeros((T, n)); Ut = np.zeros((T, m))
    Vt = np.zeros((T, m))
    At = np.zeros(T); vx = np.zeros(T, bool); vu = np.zeros(T, bool)
    ok = True;  n_assum = 0;  cost = 0.
    x_k = x0.copy();  z_k = x0.copy()          # z_0 = x_0  (Assumption 3)
    for k in range(T):
        zr_k, vr_k = steady_state(A, B, C_out, float(y_ref_sig[k]))
        e_k = x_k - z_k
        alpha_k = float(np.sqrt(max(float(e_k @ rpi.P_star_inv @ e_k), 0.)))
        if alpha_k > 1. + 1e-6:                  # should never happen
            n_assum += 1
        At[k] = alpha_k
        aseq = (alpha_seq(min(alpha_k, 1.), rpi.c_S, N) if homothetic
                else np.ones(N + 1))
        z_opt, v_opt, feas = ocp.solve(z_k, zr_k, vr_k, aseq)
        if not feas:
            ok = False;  break
        u_k = v_opt[0] + K @ e_k
        Xt[k] = x_k;  Zt[k] = z_k;  Ut[k] = u_k;  Vt[k] = v_opt[0]
        vx[k] = bool(np.any(Hx @ x_k > h_x + 1e-6))
        vu[k] = bool(np.any(Hu @ u_k > h_u + 1e-6))
        dx = x_k - zr_k;  du = u_k - vr_k
        cost += float(dx @ Q_c @ dx + du @ R_c @ du)
        w_k = sample_w(rng, unc.Sigma_w, Gw_inv)
        x_k = A_hat @ x_k + B_hat @ u_k + w_k
        z_k = A @ z_k + B @ v_opt[0]
    Te = T if ok else k
    return SimResult(x=Xt[:Te], z=Zt[:Te], u=Ut[:Te], v=Vt[:Te],
                     alpha=At[:Te],
                     y_ref=y_ref_sig[:Te], viol_x=vx[:Te], viol_u=vu[:Te],
                     n_assum=n_assum, cost=cost, ok=ok)


def time_to_lane(res, y_target, T_lc, Ts, tol):
    idx = np.where(np.abs(res.x[T_lc:, 0] - y_target) <= tol)[0]
    return (T_lc + idx[0])*Ts if len(idx) else np.nan


import io, contextlib

# ══════════════════════════════════════════════════════════════════════════════
#  INTERACTIVE EXTENSIONS  (not part of the benchmark)
#
#  Same design and the same two controllers as above, parametrised - as the MAIN
#  of the benchmark is - by the BOUNDS of Assumption 1:
#     |Delta_v_x| <= dvx_max_kmh                     (speed error bound, km/h)
#     w_k in Ell(Gamma_w),  Gamma_w = diag(w_max^2)  (additive disturbance bound)
#  The shapes Gamma_A, Gamma_B follow from the first-order ZOH sensitivities
#     Gamma_A = dv_max^2 (vec(D_A) vec(D_A)' + eta I),  same for B,  eta = 1e-4
#  (eta regularises the rank-one shape) and are then inflated by cover_factors()
#  so that the EXACT ZOH matrices satisfy the bound.  This is the construction
#  of the benchmark (there written as Gamma = chi2*Sigma, sigma_vx = dv_max /
#  sqrt(chi2)), so the default settings reproduce the paper's run.  As in the
#  benchmark, the simulated plant runs at the worst case v_x + dvx_max.
# ══════════════════════════════════════════════════════════════════════════════

from typing import List, Optional, Tuple

ETA_REG  = 1e-4           # relative regularisation of the rank-one shapes
# Two fixed scaling constants.  (i) The design depends only on the shapes Gamma;
# they are split as Gamma = kappa * Sigma purely so that the SDP of Lemma 2 is
# solved at the same numerical scale as in the benchmark (Q_max is positively
# homogeneous, so the result is independent of kappa).  (ii) Simulating one
# realisation needs a distribution inside the bound: w_k is drawn from the
# Gaussian N(0, Gamma_w/kappa_w) truncated to Ell(Gamma_w).
KAPPA_A  = float(stats.chi2.ppf(0.99, df=1))              # 6.635
KAPPA_W  = float(stats.chi2.ppf(0.99, df=4))              # 13.277

V_KMH_DEFAULT  = 60.0                                     # km/h  (paper speed)
DVX_KMH_DEFAULT = 1.0                                     # km/h  (paper bound)
W_MAX_DEFAULT  = np.sqrt(KAPPA_W*np.array([1e-4, 2e-5, 1e-4, 5e-5])*2)
#                      = [0.0515, 0.0230, 0.0515, 0.0364]  [m, rad, m/s, rad/s]
R2D = 180./np.pi


class BoundedUncertainty(UncertaintyModel):
    """Assumption 1 given directly by the shapes Gamma_A, Gamma_B, Gamma_w.

    The base class multiplies its covariances by the chi2 factors, so passing
    Sigma = Gamma/kappa together with chi2 = kappa reproduces the shapes exactly
    (Q_max is positively homogeneous: kappa * Q_max(Sigma) = Q_max(kappa Sigma)).

    `Sigma_w` is NOT used by the design - the design sees only `Gamma_w`.  It is
    the covariance of the truncated Gaussian from which `sample_w` draws the
    realisation of w_k inside the bound."""
    def __init__(self, Gamma_A, Gamma_B, Gamma_w, n, m):
        self.Sigma_DA = np.atleast_2d(Gamma_A)/KAPPA_A
        self.Sigma_DB = np.atleast_2d(Gamma_B)/KAPPA_A
        self.n, self.m = n, m
        self.chi2_DA = self.chi2_DB = KAPPA_A
        self.chi2_w  = KAPPA_W
        self.Gamma_w = np.atleast_2d(Gamma_w)
        self.Sigma_w = self.Gamma_w/KAPPA_W        # sampling only (see above)


# ── Offline design ────────────────────────────────────────────────────────────

@dataclass
class Design:
    A: np.ndarray;  B: np.ndarray;  K: np.ndarray;  C_out: np.ndarray
    Hx: np.ndarray; h_x: np.ndarray; Hu: np.ndarray; h_u: np.ndarray
    x_bounds: np.ndarray; u_bounds: np.ndarray
    Q_c: np.ndarray; R_c: np.ndarray; P_f: np.ndarray
    unc: BoundedUncertainty; rpi: RPIData
    s_x: np.ndarray; s_u: np.ndarray; term: TerminalSet
    ocp_p: TubeOCP; ocp_r: TubeOCP
    N: int; Ts: float; vx0: float
    dvx_max_kmh: float; w_max: np.ndarray
    lane_width: float; y_ref_max: float
    log: str = ""

    @property
    def dv_max(self) -> float:
        """Speed-error bound in m/s."""
        return self.dvx_max_kmh/3.6

    @property
    def s_ey(self) -> float:
        """Half-width of the RPI ellipsoid Omega in the e_y direction."""
        return float(np.sqrt(self.rpi.P_star[0, 0]))

    @property
    def u_margin(self) -> float:
        """Steering left to the nominal input by the rigid tightening [rad]."""
        return float(self.u_bounds[0] - self.s_u[0])


def design(v_kmh: float = V_KMH_DEFAULT, dvx_max_kmh: float = DVX_KMH_DEFAULT,
           w_ey: float = float(W_MAX_DEFAULT[0]), N: int = 25, Ts: float = 0.1,
           ey_max: float = 3.0, lane_width: float = 3.5, verbose: bool = False) -> Design:
    """Offline part of the benchmark, driven by the bounds of Assumption 1.

    v_kmh       : speed the model is linearised at                     [km/h]
    dvx_max_kmh : bound on the speed error, |Delta_v_x| <= dvx_max_kmh  [km/h]
    w_ey        : e_y semi-axis of the disturbance ellipsoid Ell(Gamma_w) [m]
                  (the other components scale with it)"""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        vx0    = v_kmh/3.6
        dv_max = dvx_max_kmh/3.6
        n, m = 4, 1
        A, B = discretize(vx0, Ts)
        C_out = np.array([[1., 0., 0., 0.]])

        # ancillary gain (Assumption 2)
        Q_lqr = np.diag([50., 10., 1., 10.]);  R_lqr = np.diag([1.0])
        P_lqr = la.solve_discrete_are(A, B, Q_lqr, R_lqr)
        K     = -la.solve(R_lqr + B.T @ P_lqr @ B, B.T @ P_lqr @ A)
        A_K   = A + B @ K
        assert float(np.max(np.abs(la.eigvals(A_K)))) < 1. and abs(la.det(A_K)) > 1e-9

        # uncertainty shapes from the bounds, then the exact-model cover
        D_A, D_B = d_dvx(vx0, Ts)
        dA, dB   = D_A.flatten(), D_B.flatten()
        Gamma_A  = dv_max**2*(np.outer(dA, dA) + ETA_REG*np.eye(n*n))
        Gamma_B  = dv_max**2*(np.outer(dB, dB) + ETA_REG*np.eye(n*m))
        kap_A, kap_B, _, _ = cover_factors(vx0, Ts, A, B, Gamma_A, Gamma_B, dv_max)
        Gamma_A, Gamma_B = kap_A*Gamma_A, kap_B*Gamma_B
        w_max    = (w_ey/float(W_MAX_DEFAULT[0]))*W_MAX_DEFAULT
        Gamma_w  = np.diag(w_max**2)

        # constraints, costs, terminal cost
        x_bounds = np.array([ey_max, 0.30, 2.0, 1.5]);  u_bounds = np.array([0.20])
        Hx = np.vstack([np.eye(n), -np.eye(n)]);  h_x = np.tile(x_bounds, 2)
        Hu = np.vstack([np.eye(m), -np.eye(m)]);  h_u = np.tile(u_bounds, 2)
        Q_c = np.diag([20., 5., 0.5, 0.5]);  R_c = np.diag([0.1])
        P_f = la.solve_discrete_lyapunov(A_K.T, Q_c + K.T @ R_c @ K);  P_f = 0.5*(P_f + P_f.T)

        # RPI ellipsoid, tightening
        unc = BoundedUncertainty(Gamma_A, Gamma_B, Gamma_w, n, m)
        QA, QB = compute_Qmax(unc, Hx, h_x, Hu, h_u)
        rpi = compute_rpi(A_K, unc, QA, QB)
        s_x, s_u = support_values(K, rpi.P_star, Hx, Hu)
        if not (np.all(s_x < h_x) and np.all(s_u < h_u)):
            raise ValueError("Assumption 3 violated: the bounds are too large for this "
                             "vehicle - Omega no longer fits inside the constraints.")
        # admissible set-points (the e_y row is the binding one on a straight road)
        y_ref_max = float(ey_max - s_x[0] - 0.10)
        if y_ref_max <= 0.:
            raise ValueError("no admissible set-point left: reduce the bounds.")

        # terminal set: G depends only on A_K and t*, so one set covers every
        # set-point; t* is taken as the max over a grid spanning the admissible range
        grid = np.linspace(-y_ref_max, y_ref_max, 7)
        term = TerminalSet(A_K, K, Hx, h_x, Hu, h_u, s_x, s_u,
                           [steady_state(A, B, C_out, y) for y in grid])
        # the two schemes share the same QP; one instance each, as in the benchmark
        ocp_p = TubeOCP(A, B, Hx, h_x, Hu, h_u, s_x, s_u, term, N, Q_c, R_c, P_f, "HT-MPC")
        ocp_r = TubeOCP(A, B, Hx, h_x, Hu, h_u, s_x, s_u, term, N, Q_c, R_c, P_f, "RT-MPC")

    d = Design(A=A, B=B, K=K, C_out=C_out, Hx=Hx, h_x=h_x, Hu=Hu, h_u=h_u,
               x_bounds=x_bounds, u_bounds=u_bounds, Q_c=Q_c, R_c=R_c, P_f=P_f,
               unc=unc, rpi=rpi, s_x=s_x, s_u=s_u, term=term, ocp_p=ocp_p, ocp_r=ocp_r,
               N=N, Ts=Ts, vx0=vx0, dvx_max_kmh=dvx_max_kmh, w_max=w_max,
               lane_width=lane_width, y_ref_max=y_ref_max, log=buf.getvalue())
    if verbose:
        print(d.log, end="")
        print(f"  v_x0={v_kmh:.0f} km/h  |dv_x|<={dvx_max_kmh:.2f} km/h  "
              f"w bound={np.round(w_max, 4)}\n"
              f"  Omega half-width in e_y: {d.s_ey:.4f} m   c_S={rpi.c_S:.6f}\n"
              f"  rigid steering bound: {d.u_margin*R2D:.2f} deg of {d.u_bounds[0]*R2D:.2f}   "
              f"|e_y| set-point limit: {d.y_ref_max:.2f} m")
    return d


# ── Manoeuvre generators ──────────────────────────────────────────────────────

MANOEUVRES = ["Single lane change (paper)", "Slalom", "Custom waypoints"]
PRESETS = {"Single lane change (paper)": dict(t_lc=1.2, T_sim=5.0),
           "Slalom":                     dict(t_lc=1.0, t_hold=1.5, T_sim=9.0),
           "Custom waypoints":           dict(T_sim=5.0)}


def parse_waypoints(text: str) -> List[Tuple[float, float]]:
    """'0:-1.75; 1.2:1.75' -> [(0.0, -1.75), (1.2, 1.75)]"""
    pts = []
    for tok in text.replace(",", ";").split(";"):
        tok = tok.strip()
        if not tok:
            continue
        try:
            t, y = tok.split(":")
            pts.append((float(t), float(y)))
        except ValueError:
            raise ValueError(f"cannot parse waypoint '{tok}' - expected 't:y'") from None
    if not pts:
        raise ValueError("no waypoints given")
    pts.sort()
    if pts[0][0] > 0.:
        pts.insert(0, (0., pts[0][1]))
    return pts


def waypoints_for(mode, t_lc=1.2, t_hold=1.5, y_lane=1.75, T_sim=5.0, custom=""):
    y_right, y_left = -y_lane, +y_lane
    if mode == MANOEUVRES[0]:
        return [(0., y_right), (t_lc, y_left)]
    if mode == MANOEUVRES[1]:
        pts, t, y = [(0., y_right)], t_lc, y_left
        while t < T_sim:
            pts.append((t, y));  t += t_hold;  y = -y
        return pts
    if mode == MANOEUVRES[2]:
        return parse_waypoints(custom)
    raise ValueError(f"unknown manoeuvre '{mode}'")


def build_reference(waypoints, T_sim, Ts, y_max=None) -> np.ndarray:
    """Piecewise-constant e_y set-point on the grid k*Ts."""
    T = int(round(T_sim/Ts))
    t = np.arange(T)*Ts
    y = np.full(T, waypoints[0][1], float)
    for (t_i, y_i) in waypoints[1:]:
        y[t >= t_i - 1e-9] = y_i
    if y_max is not None and np.max(np.abs(y)) > y_max + 1e-9:
        raise ValueError(f"|e_y| set-point must stay within +/-{y_max:.2f} m "
                         f"(got {np.max(np.abs(y)):.2f} m) - lower the bounds or the offsets.")
    return y


# ── Closed loop: the benchmark's run_mpc for both schemes ─────────────────────

def simulate(d: Design, y_ref_sig, plant_dvx_kmh=None, noise_seed=7):
    """Both controllers on the same plant and the same w_k realisation.
    The plant runs at v_x + plant_dvx_kmh; None = the worst case +dvx_max_kmh,
    as in the benchmark."""
    dvx = d.dv_max if plant_dvx_kmh is None else plant_dvx_kmh/3.6
    if abs(dvx) > d.dv_max + 1e-12:
        raise ValueError(f"the plant's speed error {dvx*3.6:+.2f} km/h is outside the "
                         f"bound of +/-{d.dvx_max_kmh:.2f} km/h")
    A_hat, B_hat = discretize(d.vx0 + dvx, d.Ts)
    x0 = np.array([y_ref_sig[0], 0., 0., 0.])
    common = (d.A, d.B, A_hat, B_hat, d.K, d.C_out, d.Hx, d.h_x, d.Hu, d.h_u, d.unc, d.rpi)
    res_p = run_mpc(*common, d.ocp_p, y_ref_sig, x0, d.N, d.Q_c, d.R_c,
                    homothetic=True,  rng_seed=noise_seed)
    res_r = run_mpc(*common, d.ocp_r, y_ref_sig, x0, d.N, d.Q_c, d.R_c,
                    homothetic=False, rng_seed=noise_seed)
    return res_p, res_r, dvx*3.6


def run(d: Design, mode=MANOEUVRES[0], T_sim=None, custom="", y_lane=1.75,
        plant_dvx_kmh=None, noise_seed=7):
    """Build the reference of `mode` and simulate both controllers on it.
    Returns res_p, res_r, the reference and the plant's speed error [km/h]."""
    pre = PRESETS.get(mode, {})
    T_sim = pre.get("T_sim", 5.0) if T_sim is None else T_sim
    wps = waypoints_for(mode, pre.get("t_lc", 1.2), pre.get("t_hold", 1.5),
                        y_lane, T_sim, custom)
    y_ref_sig = build_reference(wps, T_sim, d.Ts, y_max=d.y_ref_max)
    res_p, res_r, dvx_kmh = simulate(d, y_ref_sig, plant_dvx_kmh, noise_seed)
    return res_p, res_r, y_ref_sig, dvx_kmh


def summary(res) -> dict:
    T = len(res.x)
    return dict(status="OK" if res.ok else f"infeasible at k = {T}", ok=res.ok, steps=T,
                cost=float(res.cost),
                max_delta=float(np.abs(res.u).max()) if T else np.nan,          # rad
                max_v0_deg=float(np.abs(res.v).max()*R2D) if T else np.nan,     # deg
                mean_alpha=float(res.alpha.mean()) if T else np.nan,
                viol_x=int(res.viol_x.sum()), viol_u=int(res.viol_u.sum()),
                n_assum=int(res.n_assum))

# ── Animated version of the paper figure (fig_lane_change_v2) ─────────────────
#  Three panels, as in the benchmark: (a) lateral position, (b) nominal steering
#  v_{0|k} with the admissible set of the CURRENT step for each scheme
#     HT-MPC: Hu v <= h_u - alpha_k s_u      RT-MPC: Hu v <= h_u - s_u
#  and the physical limit, (c) the closed-loop scale alpha_k.  At frame k
#  everything is drawn up to step k.

_PALETTES = {
    "light": dict(p=C["p"], r=C["r"], ref="#1D1D1D", constr=C["constr"],
                  template="simple_white", axis="#000000", font="#000000",
                  set_p=0.15, set_r=0.22, mid=0.35),
    "dark":  dict(p="#3DDC97", r="#FF9F43", ref="#F2F2F2", constr="#FF6B6B",
                  template="plotly_dark", axis="#CCCCCC", font="#E6E6E6",
                  set_p=0.22, set_r=0.30, mid=0.45),
}
FONT_FAMILY = "Times New Roman, Times, Nimbus Roman, serif"


def _palette(theme):
    return _PALETTES["dark" if str(theme).lower() == "dark" else "light"]


def _rgba(hexcol, a):
    h = hexcol.lstrip("#")
    return f"rgba({int(h[0:2],16)},{int(h[2:4],16)},{int(h[4:6],16)},{a})"


def _stairs(x, y, dx):
    """Vertices of a post-step line through (x, y), drawn up to x[-1] + dx
    (the plotly equivalent of plt.step(*_ext(x, y, dx), where="post"))."""
    if len(x) == 0:
        return np.array([]), np.array([])
    xe = np.append(x, x[-1] + dx)
    return np.repeat(xe, 2)[1:-1], np.repeat(y, 2)


def _band(x, lo, hi, dx):
    """Closed polygon of fill_between(x, lo, hi, step="post")."""
    xu, yu = _stairs(x, hi, dx);  xl, yl = _stairs(x, lo, dx)
    return np.concatenate([xu, xl[::-1]]), np.concatenate([yu, yl[::-1]])


def animate(d: Design, res_p, res_r, y_ref_sig, title=None, frame_ms=100,
            height: Optional[int] = None, theme: str = "light"):
    """Plotly animation of the paper figure: e_y, v_{0|k} with the admissible
    sets of both schemes, and alpha_k."""
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    HX = _palette(theme)
    Ts, vx0 = d.Ts, d.vx0
    ey_max, lane = float(d.x_bounds[0]), d.lane_width
    dx = Ts*vx0
    T_full = len(y_ref_sig)
    # one extra frame after a failure, so that the failure is announced
    T_max = min(max(len(res_p.x) + (not res_p.ok), len(res_r.x) + (not res_r.ok), 1), T_full)
    x_end = T_full*dx
    Hu, h_u, s_u = d.Hu, d.h_u, d.s_u
    i_up = int(np.argmax(Hu[:, 0]));  i_lo = int(np.argmin(Hu[:, 0]))

    def _vset(a):                       # admissible v_{0|k} for tightening scale a [deg]
        return ((h_u[i_lo] - a*s_u[i_lo])/Hu[i_lo, 0]*R2D,
                (h_u[i_up] - a*s_u[i_up])/Hu[i_up, 0]*R2D)

    d_lo, d_up = _vset(0.)              # physical limit
    if height is None:
        height = 640

    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.035,
                        row_heights=[1., 1.1, 0.65])
    line = lambda col, w=1.8, dash="solid": dict(color=col, width=w, dash=dash)

    # ── static: constraints, lane centre, reference ──────────────────────────
    gap = lambda y1, y2: dict(x=[0, x_end, None, 0, x_end], y=[y1, y1, None, y2, y2])
    fig.add_trace(go.Scatter(**gap(ey_max, -ey_max), mode="lines", line=line(HX["constr"], 1.6),
                             name="Constraint", legendgroup="constr", legendrank=6,
                             hoverinfo="skip"), row=1, col=1)
    fig.add_trace(go.Scatter(x=[0, x_end], y=[0, 0], mode="lines",
                             line=line(HX["constr"], 1., "dot"), opacity=HX["mid"],
                             showlegend=False, hoverinfo="skip"), row=1, col=1)
    fig.add_trace(go.Scatter(**gap(d_up, d_lo), mode="lines", line=line(HX["constr"], 1.6),
                             legendgroup="constr", showlegend=False, hoverinfo="skip"),
                  row=2, col=1)
    xr_, yr_ = _stairs(np.arange(T_full)*dx, y_ref_sig, dx)
    fig.add_trace(go.Scatter(x=xr_, y=yr_, mode="lines", line=line(HX["ref"], 1.1),
                             name="Reference", legendrank=1, hoverinfo="skip"), row=1, col=1)

    # ── animated traces (in drawing order: sets, then HT, then RT on top) ────
    anim_idx = {}

    def _add(key, trace, row):
        anim_idx[key] = len(fig.data);  fig.add_trace(trace, row=row, col=1)

    for key, rank, alpha in (("p", 4, HX["set_p"]), ("r", 5, HX["set_r"])):
        _add(f"{key}_set", go.Scatter(x=[], y=[], mode="lines", fill="toself",
                                      fillcolor=_rgba(HX[key], alpha), line=dict(width=0),
                                      name=f"admissible v<sub>0|k</sub> "
                                           f"({'HT' if key == 'p' else 'RT'})",
                                      legendrank=rank, hoverinfo="skip"), row=2)
    for key, rank in (("p", 2), ("r", 3)):
        dash = "solid" if key == "p" else "dash"
        _add(f"{key}_path", go.Scatter(x=[], y=[], mode="lines", line=line(HX[key], 1.8, dash),
                                       name=LABELS[key], legendgroup=key, legendrank=rank,
                                       hovertemplate=f"{LABELS[key]}<br>x = %{{x:.1f}} m"
                                                     f"<br>e<sub>y</sub> = %{{y:.3f}} m"
                                                     f"<extra></extra>"), row=1)
        _add(f"{key}_head", go.Scatter(x=[], y=[], mode="markers",
                                       marker=dict(color=HX[key], size=8,
                                                   line=dict(color=HX["axis"], width=0.8)),
                                       legendgroup=key, showlegend=False, hoverinfo="skip"), row=1)
        _add(f"{key}_viol", go.Scatter(x=[], y=[], mode="markers",
                                       marker=dict(symbol="x", size=10, color=HX[key],
                                                   line=dict(width=2)),
                                       legendgroup=key, showlegend=False, hoverinfo="skip"), row=1)
        _add(f"{key}_v", go.Scatter(x=[], y=[], mode="lines", line=line(HX[key], 1.8, dash),
                                    legendgroup=key, showlegend=False,
                                    hovertemplate=f"{LABELS[key]}<br>x = %{{x:.1f}} m"
                                                  f"<br>v<sub>0|k</sub> = %{{y:.2f}} deg"
                                                  f"<extra></extra>"), row=2)
        _add(f"{key}_alpha", go.Scatter(x=[], y=[], mode="lines", line=line(HX[key], 1.8, dash),
                                        legendgroup=key, showlegend=False,
                                        hovertemplate=f"{LABELS[key]}<br>x = %{{x:.1f}} m"
                                                      f"<br>α = %{{y:.3f}}<extra></extra>"), row=3)

    def _upto(res, k):
        """Data of `res` up to step k, the tightening scale it USED at each step
        (alpha_k for HT-MPC, 1 for RT-MPC) and its admissible steering set."""
        kk = min(k, len(res.x) - 1)
        xs = np.arange(kk + 1)*dx
        a = (np.clip(res.alpha[:kk + 1], 0., 1.) if res is res_p else np.ones(kk + 1))
        lo, up = _vset(a)
        return kk, xs, a, lo, up

    def _frame(k):
        data, idx = [], []
        for key, res in (("p", res_p), ("r", res_r)):
            kk, xs, a, lo, up = _upto(res, k)
            bx, by = _band(xs, lo, up, dx)
            data.append(go.Scatter(x=bx, y=by))
            idx.append(anim_idx[f"{key}_set"])
        for key, res in (("p", res_p), ("r", res_r)):
            kk, xs, a, lo, up = _upto(res, k)
            vi = np.where(res.viol_x[:kk + 1])[0]
            xv, yv = _stairs(xs, res.v[:kk + 1, 0]*R2D, dx)
            xa, ya = _stairs(xs, a, dx)
            data += [go.Scatter(x=xs, y=res.x[:kk + 1, 0]),
                     go.Scatter(x=xs[-1:], y=res.x[kk:kk + 1, 0]),
                     go.Scatter(x=xs[vi], y=res.x[vi, 0]),
                     go.Scatter(x=xv, y=yv),
                     go.Scatter(x=xa, y=ya)]
            idx += [anim_idx[f"{key}_{s}"] for s in ("path", "head", "viol", "v", "alpha")]
        return data, idx

    def _title(k):
        parts = [f"k = {k:3d}   t = {k*Ts:4.1f} s   x = {k*dx:6.1f} m"]
        if len(res_p.x):
            kp = min(k, len(res_p.x) - 1)
            parts.append(f"α<sub>k</sub> = {min(res_p.alpha[kp], 1.):.3f}")
        for key, res in (("p", res_p), ("r", res_r)):
            if not res.ok and k >= len(res.x):
                parts.append(f"<b style='color:{HX[key]}'>{LABELS[key]} infeasible at "
                             f"k = {len(res.x)}</b>")
        if title:
            parts.insert(0, title)
        return "   |   ".join(parts)

    frames = []
    for k in range(T_max):
        data, idx = _frame(k)
        frames.append(go.Frame(data=data, traces=idx, name=str(k),
                               layout=go.Layout(title_text=_title(k))))
    fig.frames = frames
    d0, i0 = _frame(0)
    for tr_, i_ in zip(d0, i0):
        fig.data[i_].x = tr_.x;  fig.data[i_].y = tr_.y

    lab_every = 5 if T_max <= 80 else 10
    steps = [dict(method="animate", label=(f"{k*Ts:.1f}" if k % lab_every == 0 else ""),
                  args=[[str(k)], dict(mode="immediate", frame=dict(duration=0, redraw=False),
                                       transition=dict(duration=0))])
             for k in range(T_max)]
    fig.update_layout(
        height=height, template=HX["template"],
        font=dict(family=FONT_FAMILY, size=13, color=HX["font"]),
        title=dict(text=_title(0), x=0.01, xanchor="left", y=0.985, yanchor="top",
                   font=dict(size=13)),
        # three entries per row, as in the paper figure
        legend=dict(orientation="h", x=0.5, xanchor="center", y=1.0, yanchor="bottom",
                    entrywidth=0.3, entrywidthmode="fraction", font=dict(size=13),
                    bgcolor="rgba(0,0,0,0)"),
        margin=dict(l=70, r=20, t=110, b=120),
        updatemenus=[dict(type="buttons", showactive=False, x=0., xanchor="left",
                          y=-0.115, yanchor="top", pad=dict(t=0, r=10), direction="left",
                          buttons=[
                              dict(label="▶ Play", method="animate",
                                   args=[None, dict(frame=dict(duration=frame_ms, redraw=False),
                                                    fromcurrent=True, transition=dict(duration=0))]),
                              dict(label="❚❚ Pause", method="animate",
                                   args=[[None], dict(mode="immediate",
                                                      frame=dict(duration=0, redraw=False),
                                                      transition=dict(duration=0))])])],
        sliders=[dict(active=0, steps=steps, x=0.2, len=0.8, y=-0.1, yanchor="top",
                      pad=dict(t=0, b=0), ticklen=3, minorticklen=0, font=dict(size=10),
                      currentvalue=dict(prefix="t = ", suffix=" s", visible=True,
                                        xanchor="left", font=dict(size=12)))])
    axis = dict(showgrid=False, zeroline=False, showline=True, linewidth=1,
                linecolor=HX["axis"], mirror=True, ticks="outside", ticklen=4)
    for r_ in (1, 2, 3):
        fig.update_xaxes(range=[0, x_end], row=r_, col=1, **axis)
    fig.update_xaxes(title_text="Longitudinal distance [m]", row=3, col=1)
    fig.update_yaxes(axis)
    fig.update_yaxes(title_text="e<sub>y</sub> [m]", range=[-ey_max - 0.3, ey_max + 0.3],
                     tickvals=[-lane/2, lane/2], ticktext=[f"−{lane/2:.2f}", f"{lane/2:.2f}"],
                     row=1, col=1)
    fig.update_yaxes(title_text="v<sub>0|k</sub> [deg]", range=[d_lo - 1.5, d_up + 1.5],
                     tickvals=[-10., 0., 10.], ticktext=["−10", "0", "10"], row=2, col=1)
    fig.update_yaxes(title_text="α<sub>k</sub>", range=[0., 1.15], tickvals=[0., 1.],
                     row=3, col=1)
    fig.update_yaxes(fixedrange=True)
    return fig

  figure fonts: Times clone + CM mathtext


In [2]:
import ipywidgets as W
from IPython.display import display, clear_output, HTML

MAX_SEG = 12                                    # max segments of the custom reference
designs = {}                                    # offline designs, cached per (v, dv, w)

# values offered by the two bound sliders; the paper's values are the defaults
DV_LEVELS = [0.25, 0.50, 0.75, DVX_KMH_DEFAULT, 1.25, 1.50, 1.75]
W_LEVELS  = [0.005, 0.010, 0.020, 0.030, 0.040, float(W_MAX_DEFAULT[0]), 0.060, 0.065, 0.070]
_lbl = lambda v, ref, f: (f"{v:{f}} (paper)" if abs(v - ref) < 1e-9 else f"{v:{f}}")

# ── dark-mode styling for the whole widget area (toggled with a CSS class) ───
display(HTML("""<style>
.htmpc-box { padding: 10px 12px; border-radius: 8px; }
.htmpc-dark { background: #111111; color: #e6e6e6; }
.htmpc-dark .widget-label, .htmpc-dark .widget-readout, .htmpc-dark .widget-html-content,
.htmpc-dark .widget-checkbox label, .htmpc-dark .widget-inline-hbox .widget-label { color: #e6e6e6 !important; }
.htmpc-dark .widget-readout { background: #222 !important; border-color: #444 !important; }
.htmpc-dark input, .htmpc-dark select, .htmpc-dark .widget-dropdown select {
  background: #222 !important; color: #e6e6e6 !important; border-color: #444 !important; }
.htmpc-cell input { text-align: center; }
</style>"""))

# ── row 1: manoeuvre | T_sim ─────────────────────────────────────────────────
w_mode = W.Dropdown(options=MANOEUVRES, value=MANOEUVRES[0], description="Manoeuvre",
                    style={"description_width": "75px"}, layout=W.Layout(width="330px"))
w_tsim = W.FloatSlider(5.0, min=2.0, max=15.0, step=0.5, description="T_sim [s]",
                       style={"description_width": "65px"}, layout=W.Layout(width="300px"),
                       readout_format=".1f")
# ── row 2: the operating point and the bounds of Assumption 1 ───────────────
w_v  = W.IntSlider(int(V_KMH_DEFAULT), min=40, max=95, step=5, description="speed [km/h]",
                   style={"description_width": "100px"}, layout=W.Layout(width="300px"))
w_dv = W.SelectionSlider(options=[(_lbl(v, DVX_KMH_DEFAULT, ".2f"), v) for v in DV_LEVELS],
                         value=DVX_KMH_DEFAULT, description="Δv_x bound [km/h]",
                         style={"description_width": "125px"}, layout=W.Layout(width="335px"))
w_w  = W.SelectionSlider(options=[(_lbl(v, float(W_MAX_DEFAULT[0]), ".3f"), v) for v in W_LEVELS],
                         value=float(W_MAX_DEFAULT[0]), description="w bound (e_y) [m]",
                         style={"description_width": "130px"}, layout=W.Layout(width="340px"))

# ── custom reference: one column per segment, rows = duration / e_y ─────────
_default_y = [-1.75, 1.75, 0.0, -1.75, 1.75, 0.0, -1.75, 1.75, 0.0, -1.75, 1.75, 0.0]
CELL = W.Layout(width="80px")
w_nseg  = W.BoundedIntText(4, min=2, max=MAX_SEG, description="segments",
                           style={"description_width": "70px"}, layout=W.Layout(width="150px"))
cell_dt = [W.BoundedFloatText(1.25, min=0.1, max=15.0, step=0.1, layout=CELL) for _ in range(MAX_SEG)]
cell_y  = [W.BoundedFloatText(_default_y[i], min=-2.75, max=2.75, step=0.05, layout=CELL)
           for i in range(MAX_SEG)]
for c in cell_dt + cell_y:
    c.add_class("htmpc-cell")
grid     = W.GridBox(layout=W.Layout(grid_gap="4px 6px", align_items="center"))
tray_msg = W.HTML()
tray = W.VBox([W.HTML("<b>Custom reference</b> &mdash; each column is one segment: its duration "
                      "and the lane position e<sub>y</sub> it holds. The last duration is filled "
                      "automatically so that all segments add up to T_sim. Press <b>Run</b> when done."),
               w_nseg, grid, tray_msg])
tray.layout.display = "none"


def _rest():
    n = w_nseg.value
    return round(w_tsim.value - sum(c.value for c in cell_dt[:n-1]), 2)


def _update_last(*_):
    n, rest = w_nseg.value, _rest()
    cell_dt[n-1].value = max(rest, 0.1)
    starts = np.cumsum([0.0] + [c.value for c in cell_dt[:n-1]])
    tray_msg.value = (f"<span style='font-size:11px;color:#E05555'>the first {n-1} durations "
                      f"already exceed T_sim = {w_tsim.value:.1f} s</span>" if rest < 0.1 else
                      "<span style='font-size:11px'>segment starts [s]: " +
                      ", ".join(f"{t:.1f}" for t in starts) + "</span>")


def _build_grid(*_):
    n = w_nseg.value
    for i, c in enumerate(cell_dt[:n]):
        c.value = round(w_tsim.value/n, 1);  c.disabled = (i == n-1)
    hdr  = [W.HTML("")] + [W.HTML(f"<div style='text-align:center;font-size:11px'><b>seg {i+1}</b></div>")
                           for i in range(n)]
    row1 = [W.HTML("<span style='font-size:11px'>duration [s]</span>")] + cell_dt[:n]
    row2 = [W.HTML("<span style='font-size:11px'>e<sub>y</sub> [m]</span>")] + cell_y[:n]
    grid.children = hdr + row1 + row2
    grid.layout.grid_template_columns = f"90px repeat({n}, 84px)"
    _update_last()


def _custom_string():
    n = w_nseg.value
    starts = np.cumsum([0.0] + [c.value for c in cell_dt[:n-1]])
    return "; ".join(f"{starts[i]:.3f}:{cell_y[i].value}" for i in range(n))


def _on_mode(change):
    w_tsim.value = PRESETS[change["new"]]["T_sim"]
    tray.layout.display = "" if change["new"] == MANOEUVRES[2] else "none"


w_mode.observe(_on_mode, names="value")
w_nseg.observe(_build_grid, names="value")
w_tsim.observe(_update_last, names="value")
for c in cell_dt:
    c.observe(_update_last, names="value")
_build_grid()

# ── run row ──────────────────────────────────────────────────────────────────
w_dark = W.Checkbox(False, description="dark background", indent=False,
                    layout=W.Layout(width="160px"))
w_run  = W.Button(description="Run", button_style="success", icon="play",
                  layout=W.Layout(width="120px"))
box    = W.VBox([W.HBox([w_mode, w_tsim]), W.HBox([w_v, w_dv, w_w]), tray,
                 W.HBox([w_run, w_dark])])
box.add_class("htmpc-box")
w_dark.observe(lambda ch: (box.add_class if ch["new"] else box.remove_class)("htmpc-dark"),
               names="value")


def _show(fig, frame_ms):
    """Render the figure at cell level (never inside an Output widget, whose
    scripts some front-ends refuse to run).  Plain HTML, so exactly one figure
    appears in VS Code, JupyterLab and nbclassic; the animation is started with
    the Play button's timing (Plotly's built-in auto_play uses slow defaults)."""
    autoplay = ("Plotly.animate('{plot_id}', null, {frame: {duration: %d, redraw: false}, "
                "transition: {duration: 0}, fromcurrent: true});" % frame_ms)
    display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn",
                             auto_play=False, post_script=autoplay)))


def _on_run(_=None):
    clear_output(wait=True)
    display(box)                       # controls stay on top of the cell output
    if w_mode.value == MANOEUVRES[2] and _rest() < 0.1:
        print("Invalid manoeuvre: the segment durations exceed T_sim.");  return
    key = (w_v.value, w_dv.value, w_w.value)
    if key not in designs:                     # a few seconds: Omega, tightening, terminal set, QP
        try:
            designs[key] = design(v_kmh=w_v.value, dvx_max_kmh=w_dv.value, w_ey=w_w.value)
        except ValueError as e:
            print(f"No design exists for these bounds: {e}");  return
    d = designs[key]
    try:
        res_p, res_r, y_ref, dvx = run(d, w_mode.value, T_sim=w_tsim.value,
                                       custom=_custom_string())
    except ValueError as e:
        print(f"Invalid manoeuvre: {e}");  return
    sp, sr = summary(res_p), summary(res_r)
    ttl = (f"{w_mode.value} @ {w_v.value} km/h   |   "
           f"Ω half-width {d.s_ey:.2f} m, rigid steering bound {d.u_margin*R2D:.2f}°")
    fig = animate(d, res_p, res_r, y_ref, title=ttl, frame_ms=int(round(1000*d.Ts)),
                  theme=("dark" if w_dark.value else "light"))
    col = "#e6e6e6" if w_dark.value else "#222"
    display(HTML(
        f"<div style='font-family:monospace;font-size:12px;color:{col}'>"
        f"HT-MPC: {sp['status']}, cost {sp['cost']:.1f}, max|v<sub>0|k</sub>| {sp['max_v0_deg']:.2f}° &nbsp;|&nbsp; "
        f"RT-MPC: {sr['status']}, cost {sr['cost']:.1f}, max|v<sub>0|k</sub>| {sr['max_v0_deg']:.2f}° &nbsp;|&nbsp; "
        + (f"cost reduction <b>{100*(1-sp['cost']/sr['cost']):.1f} %</b>"
           if (sp['ok'] and sr['ok']) else "<b>not comparable (one scheme is infeasible)</b>")
        + f" &nbsp;|&nbsp; plant Δv_x = {dvx:+.2f} km/h (worst case)</div>"))
    _show(fig, int(round(1000*d.Ts)))


w_run.on_click(_on_run)
_on_run()       # draws the controls and the paper case

---

## The controls in detail

| Control | Meaning |
|---|---|
| **Manoeuvre** | *Single lane change (paper)*: right → left lane at $t = 1.2$ s. *Slalom*: alternating lane changes every 1.5 s. *Custom waypoints*: a table appears — one column per segment (2–12) with its duration and the lane position $e_y$ it holds; the last duration is filled automatically so that all segments add up to $T_{sim}$ |
| **T_sim** | simulation length [s] (set automatically when a preset is chosen) |
| **speed** | $v_{x0}$ [km/h]: the nominal speed the model is linearised at |
| **Δv_x bound** | $\lvert\Delta v_x\rvert \le \overline{\Delta v}$ [km/h]: the speed-error bound of Assumption 1. As in the benchmark, the simulated vehicle drives at the worst case $v_{x0} + \overline{\Delta v}$ |
| **w bound (e_y)** | $e_y$ semi-axis of the additive-disturbance ellipsoid $\mathrm{Ell}(\Gamma_w)$ [m]; the other components scale with it |
| **dark background** | dark theme for the figure and the whole widget area |

The figure has the three panels of the paper's figure: the lateral position $e_y$; the nominal
steering $v_{0|k}$ with the set each scheme admits for it at the current step —
$H_u v \le h_u - \alpha_k s_u$ for HT-MPC, $H_u v \le h_u - s_u$ for RT-MPC — and the physical
steering limit; and the closed-loop scale $\alpha_k$ (RT-MPC always uses $1$).

The three bound knobs default to the values of the paper (60 km/h, $\overline{\Delta v} = 1$ km/h,
the disturbance ellipsoid of the paper), and with them the playground reproduces the benchmark
run exactly: closed-loop cost 1494.3 for HT-MPC against 2347.9 for RT-MPC, a 36.4 % reduction.
Everything else — vehicle, horizon $N = 25$, constraints, costs, the plant at
$v_{x0} + \overline{\Delta v}$ and the $w_k$ realisation — is as in `LatVeh_Benchmark.ipynb`.

Larger bounds mean a larger $\Omega$, more tightening and less steering left to the nominal
input. The rigid scheme runs out of steering first: in the single lane change at
$\overline{\Delta v} = 1.25$ km/h, at 45 or 80 km/h, or at $\bar w_{e_y} = 0.060$ m — and in the
slalom already at the paper's bounds — **RT-MPC's QP becomes infeasible while HT-MPC still
completes the manoeuvre**; a little further and no design exists at all.

## What is in the collapsed cell

Three blocks, in this order:

1. a **verbatim copy** of `LatVeh_Benchmark.ipynb` — vehicle model and ZOH discretisation,
   uncertainty shapes and their exact-model check (Assumption 1), vertex bound (Lemma 2), RPI
   ellipsoid and contraction coefficient (Prop. 2, Def. 1), closed-form scale sequence
   (Prop. 3), tightening, polytopic terminal set, the tube OCP (Def. 2) and the closed loop
   (Algorithm 1);
2. `design(v_kmh, dvx_max_kmh, w_ey)` / `run(...)` / `summary(...)`, which run that same
   offline design and closed loop for a given operating point and given **bounds**;
3. `animate(...)`, the animated version of the paper's figure (`fig_lane_change_v2` in the
   benchmark).

### If the controls above do not appear

Some environments do not render `ipywidgets` (an old Colab runtime, nbviewer, a static preview
on GitHub). The same figure can always be produced without them — edit the arguments and run:

In [3]:
# Same thing without widgets: edit the arguments and run this cell.
d = design(v_kmh=60, dvx_max_kmh=DVX_KMH_DEFAULT, w_ey=float(W_MAX_DEFAULT[0]))
res_p, res_r, y_ref, dvx = run(d, "Single lane change (paper)", T_sim=5.0)
fig = animate(d, res_p, res_r, y_ref, title="Single lane change (paper) @ 60 km/h")
print(f"HT-MPC: {summary(res_p)['status']}, cost {summary(res_p)['cost']:.1f}   |   "
      f"RT-MPC: {summary(res_r)['status']}, cost {summary(res_r)['cost']:.1f}   |   "
      f"plant Δv_x = {dvx:+.2f} km/h")

from IPython.display import HTML, display
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

HT-MPC: OK, cost 1494.3   |   RT-MPC: OK, cost 2347.9   |   plant Δv_x = +1.00 km/h
